<a href="https://colab.research.google.com/github/FIGARO79/GanaBaloto/blob/main/Baloto_Analysis_with_JAX_Refactoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

if not os.path.exists('/content/drive/MyDrive/GanaBaloto'):
  print("Installing and Running For First Time.")
  %cd /content/drive/MyDrive
  !git clone https://github.com/FIGARO79/GanaBaloto.git
  !cp -r /content/drive/MyDrive/GanaBaloto
else:
  print("Repository already exist. Continue...")
  %cd /content/drive/MyDrive/GanaBaloto

Mounted at /content/drive
Repository already exist. Continue...
/content/drive/MyDrive/GanaBaloto


In [3]:
# Import necessary libraries
import pandas as pd
from IPython.display import display, HTML
import random
from itertools import combinations
import jax
import jax.numpy as jnp

# --- Configuration ---
# IMPORTANT: Update this path to the correct location of your Excel file in Google Drive
# Ensure your Google Drive is mounted in Colab (usually at /content/drive)
file_path = '/content/drive/MyDrive/GanaBaloto/baloto.xlsx'
sheet_name = 'Baloto'
columns_to_analyze = ['B1', 'B2', 'B3', 'B4', 'B5']
super_balota_column = 'SB'
prize_column = 'Premios 5+1'
num_combinations_to_generate = 10

# --- Data Loading and Preprocessing ---
try:
    # Read the Excel file
    sheet_data = pd.read_excel(file_path, sheet_name=sheet_name)

    # Drop rows with missing Super Balota values and convert to integer
    sheet_data = sheet_data.dropna(subset=[super_balota_column])
    sheet_data[super_balota_column] = sheet_data[super_balota_column].astype(int)

    # Convert main ball columns to numeric, coercing errors
    for col in columns_to_analyze:
        sheet_data[col] = pd.to_numeric(sheet_data[col], errors='coerce')
        sheet_data = sheet_data.dropna(subset=[col]) # Drop rows where conversion failed
        sheet_data[col] = sheet_data[col].astype(int) # Convert valid ones to int

    print("Data loaded and preprocessed successfully.")

except FileNotFoundError:
    print(f"Error: File not found at {file_path}. Please ensure the path is correct and Google Drive is mounted.")
    # Exit or handle the error appropriately if the file isn't found
    exit()
except KeyError as e:
    print(f"Error: Column {e} not found in the sheet '{sheet_name}'. Please check the column names.")
    exit()
except Exception as e:
    print(f"An unexpected error occurred during data loading: {e}")
    exit()


# --- Duplicate Draw Analysis ---
print("\n--- Analyzing Duplicate Draws ---")
combinations_df = sheet_data[columns_to_analyze + [super_balota_column]]
# Use .duplicated() on the relevant subset
duplicate_mask = combinations_df.duplicated(keep=False)
duplicate_combinations = sheet_data[duplicate_mask]

if duplicate_combinations.empty:
    print("No duplicate combinations found.")
else:
    print("Duplicate Combinations Found:")
    display(duplicate_combinations)


# --- Frequency Analysis ---
print("\n--- Calculating Number Frequencies ---")

# Calculate frequencies for all draws
all_draws_frequencies = {}
for column in columns_to_analyze:
    # Get top 5 most frequent numbers for each ball column
    all_draws_frequencies[column] = sheet_data[column].value_counts().head(5).index.tolist()
# Get top 5 most frequent numbers for Super Balota
all_draws_frequencies["Super Balota"] = sheet_data[super_balota_column].value_counts().head(5).index.tolist()

# Filter data for draws where the grand prize was won ('Premios 5+1' > 0)
filtered_data = sheet_data[sheet_data[prize_column] > 0].copy() # Use .copy() to avoid SettingWithCopyWarning

# Calculate frequencies for filtered draws (winners)
refined_predictions = {}
if not filtered_data.empty:
    for column in columns_to_analyze:
        refined_predictions[column] = filtered_data[column].value_counts().head(5).index.tolist()
    refined_predictions["Super Balota"] = filtered_data[super_balota_column].value_counts().head(5).index.tolist()
    print("Frequencies calculated for all draws and winning draws.")
else:
    print("No draws found with grand prize winners. Refined predictions will be empty.")
    # Handle the case where refined_predictions would be empty
    for column in columns_to_analyze:
        refined_predictions[column] = []
    refined_predictions["Super Balota"] = []


# Create DataFrames for displaying frequencies
df_all_draws = pd.DataFrame.from_dict(all_draws_frequencies, orient='index')
df_all_draws = df_all_draws.rename(columns={i: f'Top {i+1}' for i in range(df_all_draws.shape[1])})

df_refined_predictions = pd.DataFrame.from_dict(refined_predictions, orient='index')
df_refined_predictions = df_refined_predictions.rename(columns={i: f'Top {i+1}' for i in range(df_refined_predictions.shape[1])})


# --- JAX-based Probability Calculation ---
# Define the function to calculate the frequency score using JAX
# Note: This calculates a score based on historical frequency, not a strict mathematical probability.
# We apply jax.jit to compile the function for potential speedup if called repeatedly.
@jax.jit
def calculate_frequency_score_jax(combination_jax, sb_jax, b1_col_jax, b2_col_jax, b3_col_jax, b4_col_jax, b5_col_jax, sb_col_jax):
    """
    Calculates a frequency score for a given combination using JAX.

    Args:
        combination_jax (jax.Array): JAX array of the 5 main numbers.
        sb_jax (jax.Array): JAX array containing the Super Balota number.
        b1_col_jax to b5_col_jax (jax.Array): JAX arrays for each historical ball column.
        sb_col_jax (jax.Array): JAX array for the historical Super Balota column.

    Returns:
        jax.Array: The calculated frequency score.
    """
    total_draws = b1_col_jax.shape[0] # Get total draws from the length of a column
    score = 0.0 # Use float for score

    # Calculate score contribution from main balls
    ball_columns_jax = [b1_col_jax, b2_col_jax, b3_col_jax, b4_col_jax, b5_col_jax]
    for i, number in enumerate(combination_jax):
        # Count frequency of the number in the corresponding historical column
        frequency = jnp.sum(ball_columns_jax[i] == number)
        # Add the relative frequency to the score
        score += frequency / total_draws

    # Calculate score contribution from Super Balota
    sb_frequency = jnp.sum(sb_col_jax == sb_jax)
    score += sb_frequency / total_draws

    return score

# Prepare JAX arrays from the historical data ONCE for efficiency
# Convert relevant pandas columns to JAX arrays
b_cols_jax = [jnp.array(sheet_data[f'B{i+1}'].values) for i in range(5)]
sb_col_jax = jnp.array(sheet_data[super_balota_column].values)


# --- Combination Generation ---
def generate_probable_combinations(predictions, num_combinations=10):
    """
    Generates combinations based on the most frequent numbers from predictions
    and calculates their frequency scores using the JAX function.

    Args:
        predictions (dict): Dictionary containing lists of frequent numbers for each ball and SB.
                           Expected keys: 'B1'...'B5', 'Super Balota'.
        num_combinations (int): Number of combinations to generate.

    Returns:
        list: A list of tuples, where each tuple is (combination_list, sb_number, score).
              Returns an empty list if predictions are insufficient.
    """
    # Check if predictions dictionary has the required keys and non-empty lists
    required_keys = columns_to_analyze + ["Super Balota"]
    if not all(key in predictions and predictions[key] for key in required_keys):
         print("Warning: Insufficient data in 'predictions' to generate combinations.")
         return [] # Return empty list if data is missing

    # Flatten the lists of frequent numbers for main balls
    all_probable_main_numbers = [num for key in columns_to_analyze if key in predictions for num in predictions[key]]
    # Ensure unique numbers while trying to preserve some order
    unique_probable_main_numbers = sorted(list(dict.fromkeys(all_probable_main_numbers)))

    # Get the list of frequent Super Balota numbers
    probable_sb_numbers = predictions.get("Super Balota", [])

    # Ensure we have enough unique numbers to sample from
    if len(unique_probable_main_numbers) < 5 or not probable_sb_numbers:
        print("Warning: Not enough unique frequent numbers to generate combinations.")
        return []

    generated_combinations = []
    attempts = 0 # Limit attempts to avoid infinite loops if sampling is difficult
    max_attempts = num_combinations * 5 # Allow more attempts than needed

    while len(generated_combinations) < num_combinations and attempts < max_attempts:
        attempts += 1
        try:
            # Sample 5 unique numbers for the main combination
            combination = random.sample(unique_probable_main_numbers, 5)
            # Sample 1 number for the Super Balota
            sb = random.choice(probable_sb_numbers)

            # Convert combination and sb to JAX arrays for the calculation function
            combination_jax = jnp.array(combination)
            sb_jax = jnp.array(sb) # Pass SB as a JAX array (even if single element)

            # Calculate the frequency score using the JAX function
            # Pass the pre-converted historical data columns
            score = calculate_frequency_score_jax(combination_jax, sb_jax, *b_cols_jax, sb_col_jax)

            # Convert score from JAX array to Python float for storage/display
            score_float = float(score)

            # Append the result (combination as list, sb as int, score as float)
            generated_combinations.append((sorted(combination), sb, score_float))

        except ValueError as e:
            # Handle potential errors during sampling (e.g., k > population)
            print(f"Sampling error: {e}. Skipping this attempt.")
        except Exception as e:
            # Catch any other unexpected errors
            print(f"An unexpected error occurred during combination generation: {e}")

    if attempts >= max_attempts and len(generated_combinations) < num_combinations:
        print(f"Warning: Could only generate {len(generated_combinations)} combinations after {max_attempts} attempts.")

    # Sort combinations by score in descending order (highest score first)
    generated_combinations.sort(key=lambda x: x[2], reverse=True)

    return generated_combinations


# --- Manual Play Input ---
def jugada_manual():
    """
    Allows the user to input a manual combination and calculates its frequency score.

    Returns:
        tuple: (combination_list, sb_number, score_float) or None if input is invalid.
    """
    while True:
        try:
            combinacion = []
            print("\n--- Ingreso de Jugada Manual ---\n")
            # Input main numbers
            for i in range(5):
                while True:
                    try:
                        number_str = input(f"Ingrese el número {i + 1} de la combinación (1-43): \n")
                        number = int(number_str)
                        if 1 <= number <= 43:
                            if number not in combinacion:
                                combinacion.append(number)
                                break # Exit inner loop for this number
                            else:
                                print("Error: Número repetido. Ingrese un número único.")
                        else:
                            print("Error: Número fuera del rango (1-43).")
                    except ValueError:
                        print("Error: Entrada inválida. Por favor, ingrese un número entero.")

            # Input Super Balota
            while True:
                 try:
                    sb_str = input("Ingrese el número de la Super Balota (1-16): \n")
                    sb = int(sb_str)
                    if 1 <= sb <= 16:
                        break # Exit inner loop for Super Balota
                    else:
                        print("Error: Número fuera del rango (1-16).")
                 except ValueError:
                    print("Error: Entrada inválida. Por favor, ingrese un número entero.")

            # Calculate score using JAX function
            combination_jax = jnp.array(combinacion)
            sb_jax = jnp.array(sb)
            score = calculate_frequency_score_jax(combination_jax, sb_jax, *b_cols_jax, sb_col_jax)
            score_float = float(score) # Convert to float

            print(f"\nCombinación ingresada: {sorted(combinacion)}, Super Balota: {sb}")
            # Display score - Note: Higher score means more frequent numbers appeared historically
            # It's not a direct probability percentage.
            print(f"Puntaje de Frecuencia Histórica: {score_float:.4f}") # Display score with more precision

            return sorted(combinacion), sb, score_float

        except Exception as e: # Catch potential errors during the process
            print(f"Ocurrió un error durante el ingreso manual: {e}")
            return None # Indicate failure


# --- Main Execution Logic ---

# Display Frequency Tables
print("\n--- Mostrando Tablas de Frecuencia ---")
html_str_freq = f"""
<h2 style='text-align:center;'>Análisis de Frecuencia de Números Baloto</h2>
<table style='width:100%; border-collapse: collapse;'>
  <tr>
    <td style='vertical-align: top; padding: 10px; border: 1px solid #ddd;'>
      <h3>Números más frecuentes (Todos los Sorteos)</h3>
      {df_all_draws.to_html(classes='table table-striped', justify='center')}
    </td>
    <td style='vertical-align: top; padding: 10px; border: 1px solid #ddd;'>
      <h3>Números más frecuentes (Sorteos con Premio 5+1)</h3>
      {df_refined_predictions.to_html(classes='table table-striped', justify='center')}
    </td>
  </tr>
</table>
"""
display(HTML(html_str_freq))

# Manual Play Loop
while True:
    response = input("\n¿Desea ingresar una jugada manual? (s/n): \n").lower()
    if response == 's':
        manual_result = jugada_manual()
        if manual_result:
            # Optionally store or use the manual result
            pass
    elif response == 'n':
        print("\nProcediendo a generar jugadas automáticas.")
        break
    else:
        print("Respuesta no válida. Por favor, ingrese 's' o 'n'.")

# Automatic Combination Generation and Display Loop
print("\n--- Generando Combinaciones Automáticas (Basadas en Sorteos Ganadores) ---")

while True:
    # Generate combinations using refined predictions (from winning draws)
    # If refined_predictions is empty, generate_probable_combinations will handle it
    probable_combinations = generate_probable_combinations(refined_predictions, num_combinations=num_combinations_to_generate)

    if not probable_combinations:
        print("No se pudieron generar combinaciones automáticas (datos insuficientes o error).")
        # Decide if you want to try with all_draws_frequencies or stop
        print("Intentando generar combinaciones basadas en TODOS los sorteos...")
        probable_combinations = generate_probable_combinations(all_draws_frequencies, num_combinations=num_combinations_to_generate)
        if not probable_combinations:
             print("Tampoco se pudieron generar combinaciones con todos los sorteos. Finalizando.")
             break # Exit the loop if generation fails completely


    # Create DataFrame for display
    data_for_df = []
    for combinacion, sb, score in probable_combinations:
        # Format combination list as a string
        combinacion_str = ', '.join(map(str, combinacion))
        # Append data row: combination string, SB number, score
        data_for_df.append([combinacion_str, sb, score])

    # Create the pandas DataFrame
    df_combinations = pd.DataFrame(data_for_df, columns=['Combinacion', 'Super Balota', 'Puntaje Frecuencia'])
    # Set index starting from 1
    df_combinations.index = range(1, len(df_combinations) + 1)

    # Format the 'Puntaje Frecuencia' column for display (e.g., 4 decimal places)
    # DO THIS STEP ON THE PANDAS DATAFRAME
    df_combinations['Puntaje Frecuencia'] = df_combinations['Puntaje Frecuencia'].map('{:.4f}'.format)


    # Display the DataFrame as HTML
    html_str_comb = f"""
    <div style='margin-top: 20px;'>
      <h3 style='text-align:center;'>Combinaciones Automáticas Sugeridas y su Puntaje de Frecuencia</h3>
      {df_combinations.to_html(classes='table table-hover', justify='center', index=True)}
    </div>
    """
    display(HTML(html_str_comb))

    # Ask user if they want more combinations
    response = input("\n¿Desea generar MÁS combinaciones automáticas? (s/n): \n").lower()
    if response == 's':
        continue # Continue the loop to generate more
    elif response == 'n':
        print("\n¡Mucha suerte con tus números!")
        break # Exit the loop
    else:
        print("Respuesta no válida. Finalizando.")
        break # Exit on invalid input after generation

print("\n--- Análisis Finalizado ---")

Data loaded and preprocessed successfully.

--- Analyzing Duplicate Draws ---
No duplicate combinations found.

--- Calculating Number Frequencies ---
Frequencies calculated for all draws and winning draws.

--- Mostrando Tablas de Frecuencia ---


,Top 1,Top 2,Top 3,Top 4,Top 5
B1,1,2,3,6,5
B2,20,14,8,11,12
B3,25,20,22,24,26
B4,34,37,35,30,32
B5,43,42,41,40,39
Super Balota,11,7,13,16,2
,Top 1,Top 2,Top 3,Top 4,Top 5
B1,8,5,9,2,6
B2,15,9,6,11,17
B3,19,13,28,14,34



¿Desea ingresar una jugada manual? (s/n): 
s

--- Ingreso de Jugada Manual ---

Ingrese el número 1 de la combinación (1-43): 
2
Ingrese el número 2 de la combinación (1-43): 
20
Ingrese el número 3 de la combinación (1-43): 
25
Ingrese el número 4 de la combinación (1-43): 
34
Ingrese el número 5 de la combinación (1-43): 
43
Ingrese el número de la Super Balota (1-16): 
7

Combinación ingresada: [2, 20, 25, 34, 43], Super Balota: 7
Puntaje de Frecuencia Histórica: 0.4914

¿Desea ingresar una jugada manual? (s/n): 
s

--- Ingreso de Jugada Manual ---

Ingrese el número 1 de la combinación (1-43): 
8
Ingrese el número 2 de la combinación (1-43): 
17
Ingrese el número 3 de la combinación (1-43): 
23
Ingrese el número 4 de la combinación (1-43): 
35
Ingrese el número 5 de la combinación (1-43): 
42
Ingrese el número de la Super Balota (1-16): 
7

Combinación ingresada: [8, 17, 23, 35, 42], Super Balota: 7
Puntaje de Frecuencia Histórica: 0.3676

¿Desea ingresar una jugada manual? (s/n):

,Combinacion,Super Balota,Puntaje Frecuencia
1,"5, 9, 17, 23, 31",11,0.2426
2,"8, 13, 18, 35, 38",13,0.2157
3,"6, 19, 23, 27, 35",11,0.2083
4,"6, 8, 9, 14, 24",11,0.1924
5,"9, 13, 23, 34, 38",11,0.1900
6,"6, 19, 27, 28, 34",15,0.1887
7,"5, 8, 17, 31, 34",14,0.1728
8,"2, 5, 17, 23, 31",7,0.1642
9,"9, 14, 19, 34, 42",7,0.1164
10,"2, 8, 23, 34, 35",7,0.0968



¿Desea generar MÁS combinaciones automáticas? (s/n): 
s


...


¿Desea generar MÁS combinaciones automáticas? (s/n): 
s


,Combinacion,Super Balota,Puntaje Frecuencia
1,"14, 15, 28, 34, 41",7,0.3051
2,"5, 6, 11, 34, 42",14,0.2402
3,"8, 17, 24, 34, 35",14,0.2218
4,"6, 8, 9, 24, 38",11,0.2145
5,"2, 5, 11, 34, 35",15,0.2047
6,"19, 28, 31, 38, 41",7,0.1936
7,"5, 18, 24, 38, 41",14,0.1924
8,"5, 9, 17, 19, 24",14,0.1691
9,"8, 9, 14, 27, 42",11,0.1581
10,"17, 34, 35, 41, 42",15,0.1017



¿Desea generar MÁS combinaciones automáticas? (s/n): 
s


,Combinacion,Super Balota,Puntaje Frecuencia
1,"5, 17, 24, 27, 28",15,0.2475
2,"2, 17, 19, 28, 41",7,0.2267
3,"15, 17, 19, 23, 34",11,0.2071
4,"15, 18, 23, 24, 38",14,0.2034
5,"6, 24, 31, 41, 42",14,0.1691
6,"6, 13, 15, 34, 42",13,0.1654
7,"2, 5, 11, 27, 31",11,0.1605
8,"2, 13, 31, 35, 42",14,0.1250
9,"5, 6, 27, 34, 38",13,0.1213
10,"13, 17, 24, 41, 42",15,0.1017



¿Desea generar MÁS combinaciones automáticas? (s/n): 
n

¡Mucha suerte con tus números!

--- Análisis Finalizado ---
